# Intake Coding Quality — PDGM CY2026

Of the 74,719 ICD-10 codes in the CCFTF PDGM mapping, how many can actually
anchor a home health episode as a primary diagnosis?

Everything here comes from CMS source files in `data/`. No patient data.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), "..", "src"))

import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.width", 120)
TABLEAU = os.path.join("..", "outputs", "tableau")

## 1. The gates\n\nEach gate is an independent reason a code cannot serve as a primary diagnosis.

In [ ]:
gates = pd.read_csv(os.path.join(TABLEAU, "06_intake_coding_gates.csv"))
gates

In [ ]:
clean = gates.loc[gates["gate"] == "Clean as PDGM primary"].iloc[0]
print(f"Clean as PDGM primary: {clean['codes_failing']:,} codes "
      f"({clean['pct_of_all_codes']}%)")
print(f"Fail at least one gate: {74719 - clean['codes_failing']:,} codes")

## 2. Difficulty by clinical group

A "pitfall" is a code that is unspecified, is a manifestation code needing its
etiology sequenced first, or carries a code-first requirement. All three mean
rework at intake.

In [ ]:
difficulty = pd.read_csv(os.path.join(TABLEAU, "07_coding_difficulty_by_group.csv"))
difficulty.sort_values("pct_with_pitfall", ascending=False)

In [ ]:
top = difficulty.nlargest(1, "pct_with_pitfall").iloc[0]
low = difficulty.nsmallest(1, "pct_with_pitfall").iloc[0]
print(f"Highest: {top['clinical_group']} {top['pct_with_pitfall']}% "
      f"of {top['codes']:,} codes")
print(f"Lowest : {low['clinical_group']} {low['pct_with_pitfall']}%")

### Rate is not volume

MMTA-Cardiac has the worst pitfall *rate* but only 683 codes. MS Rehab has
19,200 codes at a much lower rate. Ranking on count alone would point at the
wrong group.

In [ ]:
difficulty.assign(
    share_of_all_pitfalls=lambda d: (d["codes_with_pitfall"]
                                     / d["codes_with_pitfall"].sum() * 100).round(1)
)[["clinical_group", "codes", "pct_with_pitfall",
   "codes_with_pitfall", "share_of_all_pitfalls"]].sort_values(
    "share_of_all_pitfalls", ascending=False)

## 3. The figure\n\n`make_figures.py` renders the composed report figure.

In [ ]:
import make_figures
make_figures.main()

from IPython.display import Image
Image(os.path.join("..", "outputs", "figures", "01_intake_coding_quality.png"))